# Week 8-E: Real single-promotion controller

Construct **B0-GW** from measured canonical-draft token uncertainty with five paper-grouped FIT folds. Gain weights are fixed at `1 + 5 * max(0, best_delta)` and are used only as FIT training weights. Gold outcomes never decide online actions.

Compare E0 (k4 STOP), E1 (always promote DraftSemantic), E2 (B0-GW + DraftSemantic), E3 (B0-GW + frozen U4-A), E4 only if attribution survived the W8-D closeout, and E5 HURT-veto variants. All coefficients and thresholds are chosen from FIT OOF downstream F1. Within 0.003 of the best F1, prefer lower cumulative prompt tokens, then lower cached generation latency, then simplicity.

INTERNAL-VAL is previously inspected development confirmation. Do not retune on it. This is a replay of real saved single-promotion generations, including harmful choices; **there is no oracle gate**. No DAFiR-2 or final TEST claim is made. Two-promotion evaluation requires new sequential outcomes.

Latency is labelled as a cached-generation estimate: full deployed routing/selector overhead has not been measured. Final and cumulative prompt tokens are reported separately.

In [1]:
import sys, subprocess, json
from pathlib import Path
import pandas as pd
from IPython.display import display
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / 'configs/week8/w8_dafir.yaml').is_file())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
OUTPUT = REPO_ROOT / 'results/week8/w8e_controller'
SCRIPT = REPO_ROOT / 'scripts/week8/run_w8e_controller_tuning.py'
def run(*args):
    subprocess.run([sys.executable, str(SCRIPT), *map(str, args)], cwd=REPO_ROOT, check=True)
print('Python:', sys.executable)
run('--check')

Python: /home/gunavenkat/Downloads/CS787-RAG-Project/sara_env/bin/python


fit: 1746 measured gate inputs, 10476 canonical candidates
internal_val: 342 measured gate inputs, 2052 canonical candidates
Ready: FIT-only tuning, no oracle, no mock fallback; DAFiR-1 only


## Verify W8-D closeout and real gate inputs

The original W8-B cache had INTERNAL-VAL token diagnostics only. The additional canonical token backfill supplies FIT as well and checks token alignment. Missing inputs raise an error; no mock data are manufactured. Backfill results live separately from the old W8-B files.

In [2]:
closeout = json.loads((REPO_ROOT / 'results/week8/w8d_closeout/closeout_summary.json').read_text())
print('Frozen W8-D decision:', closeout['decision'])
for split in ('fit', 'internal_val'):
    path = REPO_ROOT / f'results/week8/w8e_gate_inputs/gate_features_{split}.csv'
    frame = pd.read_csv(path)
    print(split, '| measured gate feature rows:', len(frame))
    assert len(frame) == {'fit': 1746, 'internal_val': 342}[split]
print('Gate weights and decision thresholds will use FIT only.')

Frozen W8-D decision: {'candidate_frozen_on_fit': 'S7', 'confirmed_quality': False, 'confirmed_safety': False, 'candidate_survives': False, 'online_attribution_survives': False, 'verdict': 'no demonstrated attribution benefit in tested variants', 'interpretation': 'Dependence on compressed evidence is distinct from benefit of exposing full text.', 'do_not_retune_on_internal_val': True}
fit | measured gate feature rows: 1746
internal_val | measured gate feature rows: 342
Gate weights and decision thresholds will use FIT only.


## Construct and freeze the controller

This command trains the logistic gate, evaluates the bounded controller grid on FIT OOF outcomes, saves `frozen_controller.json`, and then confirms all frozen families on INTERNAL-VAL. A completed run is reused for inspection rather than silently retuned.

In [3]:
run('--full', '--output-dir', OUTPUT)
summary = json.loads((OUTPUT / 'controller_summary.json').read_text())
print('FIT-selected policy:', summary['selected_policy'])
print('Frozen configuration:', summary['selected_config'])
print('E4 attribution included:', summary['E4_included'], '|', summary['E4_reason'])

A completed frozen run exists. Inspect controller_summary.json and saved decisions; use a new directory only for an explicitly new experiment.
Frozen selected policy: E5-DraftSemantic


FIT-selected policy: E5-DraftSemantic
Frozen configuration: {'policy_name': 'E5-DraftSemantic', 'family': 'E5', 'selector': 'S3', 'complexity': 4, 'gate_threshold': 0.3, 'tau_help': 0.0, 'tau_hurt': 0.1, 'uses_attribution': False}
E4 attribution included: False | no demonstrated attribution benefit in tested variants


## FIT OOF and development confirmation

Use actual downstream F1 and all-query HURT rates when comparing policies. HELP precision and HURT rate among promotions use a different denominator. STOP incurs only the k4 generation; PROMOTE incurs both k4 and selected promotion prompts. No true F1, HELP/HURT label, or oracle opportunity is used to choose the action.

In [4]:
cols = ['downstream_f1', 'gain_over_k4', 'gain_over_k6', 'promotion_rate',
        'help_rate_all_queries', 'hurt_rate_all_queries', 'hurt_rate_promotions',
        'mean_final_prompt_tokens', 'mean_cumulative_prompt_tokens',
        'estimated_generation_latency_sec']
print('FIT OOF:')
display(pd.DataFrame(summary['fit_metrics']).T[cols])
print('INTERNAL-VAL confirmation:')
display(pd.DataFrame(summary['internal_val_metrics']).T[cols])
print('Latency excludes unmeasured deployed routing, gate and selector overhead.')
print('Previously inspected INTERNAL-VAL is not a pristine final test set.')

FIT OOF:


,downstream_f1,gain_over_k4,gain_over_k6,promotion_rate,help_rate_all_queries,hurt_rate_all_queries,hurt_rate_promotions,mean_final_prompt_tokens,mean_cumulative_prompt_tokens,estimated_generation_latency_sec
E0,0.385631,0.0,-0.025366,0.0,0.0,0.0,0.0,940.544101,940.544101,2.698204
E1,0.405454,0.019823,-0.005543,1.0,0.143184,0.098511,0.098511,1146.769187,2087.313288,5.854044
E2,0.404339,0.018708,-0.006658,0.635739,0.132875,0.091638,0.144144,1073.623711,1670.347079,4.838071
E3,0.401424,0.015793,-0.009573,0.635739,0.132875,0.095074,0.14955,1086.483391,1683.206758,4.873118
E5-DraftSemantic,0.402719,0.017088,-0.008279,0.584765,0.098511,0.058419,0.099902,1062.339061,1614.798969,4.537084
E5-U4-A,0.401446,0.015815,-0.009551,0.603093,0.126002,0.088202,0.146249,1079.5189,1646.275487,4.729191


INTERNAL-VAL confirmation:


,downstream_f1,gain_over_k4,gain_over_k6,promotion_rate,help_rate_all_queries,hurt_rate_all_queries,hurt_rate_promotions,mean_final_prompt_tokens,mean_cumulative_prompt_tokens,estimated_generation_latency_sec
E0,0.389061,0.0,-0.012944,0.0,0.0,0.0,0.0,935.157895,935.157895,2.691779
E1,0.412572,0.023511,0.010567,1.0,0.146199,0.099415,0.099415,1133.22807,2068.385965,5.836315
E2,0.406122,0.017061,0.004117,0.625731,0.131579,0.087719,0.140187,1060.959064,1651.935673,4.790088
E3,0.401963,0.012902,-0.000042,0.625731,0.122807,0.070175,0.11215,1084.216374,1675.192982,4.840172
E5-DraftSemantic,0.40008,0.011018,-0.001926,0.421053,0.055556,0.046784,0.111111,1017.593567,1416.111111,3.931297
E5-U4-A,0.399381,0.01032,-0.002624,0.48538,0.087719,0.035088,0.072289,1050.204678,1509.903509,4.238335


Latency excludes unmeasured deployed routing, gate and selector overhead.
Previously inspected INTERNAL-VAL is not a pristine final test set.


## Decision identity and saved artifacts

Each action retains query and paper identity; promoted actions retain original passage index, semantic rank, and BM25 rank. No candidate index is hardcoded. STOP has null passage/rank fields. Candidate scores and probability files remain separate from retrospective outcome lookup.

In [5]:
selected = summary['selected_policy']
decisions = pd.read_csv(OUTPUT / f'{selected}_internal_val_decisions.csv')
required = ['question_id', 'paper_id', 'candidate_passage_index', 'semantic_rank', 'bm25_rank', 'selected_action']
assert set(required) <= set(decisions)
assert decisions.question_id.nunique() == 342
promoted = decisions.selected_action.eq('PROMOTE')
assert decisions.loc[promoted, ['candidate_passage_index', 'semantic_rank', 'bm25_rank']].notna().all().all()
assert decisions.loc[~promoted, ['candidate_passage_index', 'semantic_rank', 'bm25_rank']].isna().all().all()
assert summary['oracle_gate'] is False and summary['max_promotions'] == 1
display(decisions[required].head(12))
print('Frozen controller:', OUTPUT / 'frozen_controller.json')
print('Gate model:', OUTPUT / 'b0_gw_gate.joblib')
print('No retuning or new SARA generation is needed to inspect this completed replay.')

,question_id,paper_id,candidate_passage_index,semantic_rank,bm25_rank,selected_action
0,0,0,NaN,NaN,NaN,STOP
1,1,0,NaN,NaN,NaN,STOP
2,1008,296,NaN,NaN,NaN,STOP
3,1009,296,NaN,NaN,NaN,STOP
4,1010,296,NaN,NaN,NaN,STOP
5,1011,296,NaN,NaN,NaN,STOP
6,1020,301,NaN,NaN,NaN,STOP
7,1021,301,NaN,NaN,NaN,STOP
8,1022,301,7.0,7.0,8.0,PROMOTE
9,1039,307,NaN,NaN,NaN,STOP


Frozen controller: /home/gunavenkat/Downloads/CS787-RAG-Project/results/week8/w8e_controller/frozen_controller.json
Gate model: /home/gunavenkat/Downloads/CS787-RAG-Project/results/week8/w8e_controller/b0_gw_gate.joblib
No retuning or new SARA generation is needed to inspect this completed replay.
